# Fase dos: qué le hicieron los modelos entrenados a su representación

La fase uno dice qué método gana. No dice si ganó **alineando los dominios clase
por clase**, que es lo que las dos formulaciones afirman. Esa afirmación vive en
la representación, así que se mide ahí.

Este cuaderno no entrena nada. Carga los checkpoints que guardó la fase uno y lee
qué representan — que es lo que hace barata cualquier pregunta posterior: todo lo
que se pueda calcular a partir de pesos y datos se puede agregar cuando se le
ocurra a alguien, sin volver a correr la campaña.

Dos reglas gobiernan cada figura de abajo, y las dos existen para que un dibujo no
salga más lindo que la corrida que describe.

**Una sola semilla de exhibición, no la mediana de cada método.** Paneles sacados
de semillas distintas se diferenciarían en el método *y* en el sorteo, y como la
semilla fija la partición, ni siquiera compartirían las bolsas. La semilla se
elige con una regla que no favorece a nadie: aquella cuya exactitud media
**sobre todos los métodos** es la mediana.

**Artefactos medianos, nunca el mejor.** El mejor de N crece con la dispersión
propia del método, así que comparar mejor contra mejor le hace el regalo más
grande al método más ruidoso.

In [1]:
import os
import sys
from pathlib import Path


def find_repository() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (os.environ.get("MIL_CREDA_REPO", ""), "/content", "/kaggle/working"):
        if base and Path(base).is_dir():
            candidates.append(Path(base))
            candidates.extend(sorted(Path(base).glob("*")))
    for candidate in candidates:
        if (candidate / "src" / "MIL_CREDA_Benchmark").is_dir():
            return candidate.resolve()
    raise SystemExit(
        "cannot find the repository. Set MIL_CREDA_REPO to the checkout that "
        "holds src/MIL_CREDA_Benchmark, or run this notebook from inside it."
    )


REPOSITORY = find_repository()
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

repository: /Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation


In [2]:
import json

import torch

from MIL_CREDA_Benchmark import config, harness, latent, tables

device = harness.resolve_device()
found = latent.available()
if not found:
    raise SystemExit(
        f"no hay checkpoints en {config.MODELS}. Corré primero la fase uno: guarda "
        f"las repeticiones más cercanas a la mediana de cada celda."
    )

runs = [json.loads(line) for line in
        (config.RESULTS / "runs.jsonl").read_text().splitlines() if line.strip()]
seed = latent.display_seed(runs)
print(f"{len(found)} checkpoints en {device} · semilla de exhibición: {seed}")

/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


60 checkpoints en mps · semilla de exhibición: 0


In [3]:
readings = [latent.analyse(record, device) for record in found]
print(f"{len(readings)} checkpoints medidos")

60 checkpoints medidos


## 1 · La razón entre distancias

**Qué mide:** la distancia media entre los centroides de una misma clase en los
dos dominios, dividida por la distancia media entre clases distintas. **Por qué:**
una distancia cruda en un embedding no se puede comparar entre dos modelos —el
espacio no tiene escala fija y cada corrida se acomoda a la suya—, pero la razón
sí. Y hacen falta las dos distancias juntas: si la de arriba baja y la de abajo se
mantiene, eso es alineación condicional; si bajan las dos, el espacio colapsó, y
el primer número solo habría llamado éxito a eso. **Más bajo es mejor**, siempre
que la distancia entre clases no haya bajado también.

In [4]:
print(tables.render_readings(readings, "geometry.ratio",
                             "razón: misma clase entre dominios / entre clases"))

razón: misma clase entre dominios / entre clases  ·  1 checkpoint(s) por celda  ·  research-concept-r16.md
el ± es la dispersión entre checkpoints guardados, no entre semillas: esto mide geometría y la fase uno mide exactitud
!! 1 repetición(es): el ± de abajo es cero por construcción, no por acuerdo. Son estimaciones puntuales, no resultados.
!! piloto: el protocolo declara 30 repeticiones y 20 épocas. Nada de esto es un resultado.

Método                      M->U              U->M              M->S              S->M              U->S              S->U     Prom.
Baseline           0.598 ± 0.000     0.530 ± 0.000     1.205 ± 0.000     1.474 ± 0.000     1.205 ± 0.000     1.320 ± 0.000     1.055
CREDA*             0.586 ± 0.000     0.555 ± 0.000     1.230 ± 0.000     1.521 ± 0.000     1.195 ± 0.000     1.343 ± 0.000     1.071
CREDA              0.691 ± 0.000     0.672 ± 0.000     1.218 ± 0.000     1.780 ± 0.000     1.244 ± 0.000     1.451 ± 0.000     1.176
MIL-Baseline       0.961 ± 0.0

### 1b · Las dos distancias por separado

**Qué mide:** los dos numeradores de la razón anterior. **Por qué:** es lo único
que separa una alineación de un colapso. **Descriptivo:** se leen juntas, ninguna
gana nada por sí sola.

In [5]:
print(tables.render_readings(readings, "geometry.crossDomainSameClass",
                             "misma clase, entre los dos dominios"))
print()
print(tables.render_readings(readings, "geometry.betweenClasses",
                             "clases distintas, dentro de un dominio"))

misma clase, entre los dos dominios  ·  1 checkpoint(s) por celda  ·  research-concept-r16.md
el ± es la dispersión entre checkpoints guardados, no entre semillas: esto mide geometría y la fase uno mide exactitud
!! 1 repetición(es): el ± de abajo es cero por construcción, no por acuerdo. Son estimaciones puntuales, no resultados.
!! piloto: el protocolo declara 30 repeticiones y 20 épocas. Nada de esto es un resultado.

Método                      M->U              U->M              M->S              S->M              U->S              S->U     Prom.
Baseline          19.011 ± 0.000    17.595 ± 0.000    27.487 ± 0.000    17.856 ± 0.000    31.137 ± 0.000    18.449 ± 0.000    21.922
CREDA*            18.060 ± 0.000    17.418 ± 0.000    25.121 ± 0.000    18.287 ± 0.000    28.046 ± 0.000    18.375 ± 0.000    20.885
CREDA             20.147 ± 0.000    19.653 ± 0.000    29.423 ± 0.000    17.541 ± 0.000    32.647 ± 0.000    15.486 ± 0.000    22.483
MIL-Baseline      36.729 ± 0.000    22.070 

## 2 · Qué tan separables siguen los dominios

**Qué mide:** con qué exactitud una regla lineal todavía distingue de qué dominio
viene cada punto, por validación cruzada. **Por qué:** si la representación se
volvió invariante al dominio, esa regla no debería poder. **El azar es 0,500 y más
cerca de 0,500 es mejor** — pero, igual que la razón, no distingue alineación de
colapso por sí sola, y por eso se lee junto a la tabla 1.

In [6]:
print(tables.render_readings(readings, "domainSeparability",
                             "exactitud de un clasificador de dominio (azar 0,500)"))

exactitud de un clasificador de dominio (azar 0,500)  ·  1 checkpoint(s) por celda  ·  research-concept-r16.md
el ± es la dispersión entre checkpoints guardados, no entre semillas: esto mide geometría y la fase uno mide exactitud
!! 1 repetición(es): el ± de abajo es cero por construcción, no por acuerdo. Son estimaciones puntuales, no resultados.
!! piloto: el protocolo declara 30 repeticiones y 20 épocas. Nada de esto es un resultado.

Método                      M->U              U->M              M->S              S->M              U->S              S->U     Prom.
Baseline           0.930 ± 0.000     0.851 ± 0.000     0.947 ± 0.000     0.994 ± 0.000     0.894 ± 0.000     0.981 ± 0.000     0.933
CREDA*             0.921 ± 0.000     0.876 ± 0.000     0.954 ± 0.000     0.992 ± 0.000     0.887 ± 0.000     0.974 ± 0.000     0.934
CREDA              0.942 ± 0.000     0.901 ± 0.000     0.970 ± 0.000     0.994 ± 0.000     0.954 ± 0.000     0.986 ± 0.000     0.958
MIL-Baseline       0.890 ±

## 3 · La afirmación propia del término local

**Qué mide:** cuánta de la masa de correspondencia que un sujeto de destino
reparte sobre los sujetos de fuente cae sobre sujetos de **su clase verdadera**.
**Por qué:** es la afirmación del término local dicha como número en vez de como
dibujo. Las etiquetas verdaderas de destino se leen acá y en ningún momento del
entrenamiento: puntúan una cantidad que el método produjo sin ellas.
**Más alto es mejor**, y el azar es 1/10.

In [7]:
print(tables.render_readings(readings, "correspondence.massOnTrueClass",
                             f"masa en la clase verdadera (azar {1 / config.CLASSES:.3f})"))

masa en la clase verdadera (azar 0.100)  ·  1 checkpoint(s) por celda  ·  research-concept-r16.md
el ± es la dispersión entre checkpoints guardados, no entre semillas: esto mide geometría y la fase uno mide exactitud
!! 1 repetición(es): el ± de abajo es cero por construcción, no por acuerdo. Son estimaciones puntuales, no resultados.
!! piloto: el protocolo declara 30 repeticiones y 20 épocas. Nada de esto es un resultado.

Método                      M->U              U->M              M->S              S->M              U->S              S->U     Prom.
MIL-CREDA          0.423 ± 0.000     0.851 ± 0.000     0.144 ± 0.000     0.268 ± 0.000     0.126 ± 0.000     0.201 ± 0.000     0.335
MIL-CREDA-U        0.236 ± 0.000     0.752 ± 0.000     0.100 ± 0.000     0.252 ± 0.000     0.171 ± 0.000     0.181 ± 0.000     0.282
MIL-CREDA-A        0.415 ± 0.000     0.612 ± 0.000     0.150 ± 0.000     0.168 ± 0.000     0.158 ± 0.000     0.184 ± 0.000     0.281
MIL-CREDA-K        0.508 ± 0.000     0.

### 3b · Cuánto reparte la atención

**Qué mide:** la entropía de los pesos dentro de la bolsa, normalizada a `[0, 1]`.
**Por qué:** 1,000 sería estar pesando todas las instancias por igual, que es
exactamente la media uniforme que la atención venía a mejorar; cerca de 0 sería
apoyar el sujeto entero en unas pocas instancias. **Descriptivo:** ni alto ni bajo
es mejor por sí mismo, pero un valor pegado a 1,000 quiere decir que la atención
no está haciendo nada.

In [8]:
print(tables.render_readings(readings, "attentionSpread",
                             "entropía de los pesos dentro de la bolsa (1,000 = uniforme)"))

entropía de los pesos dentro de la bolsa (1,000 = uniforme)  ·  1 checkpoint(s) por celda  ·  research-concept-r16.md
el ± es la dispersión entre checkpoints guardados, no entre semillas: esto mide geometría y la fase uno mide exactitud
!! 1 repetición(es): el ± de abajo es cero por construcción, no por acuerdo. Son estimaciones puntuales, no resultados.
!! piloto: el protocolo declara 30 repeticiones y 20 épocas. Nada de esto es un resultado.

Método                      M->U              U->M              M->S              S->M              U->S              S->U     Prom.
MIL-Baseline       0.710 ± 0.000     0.768 ± 0.000     0.609 ± 0.000     0.704 ± 0.000     0.785 ± 0.000     0.672 ± 0.000     0.708
MIL-CREDA**        0.695 ± 0.000     0.794 ± 0.000     0.638 ± 0.000     0.803 ± 0.000     0.824 ± 0.000     0.737 ± 0.000     0.749
MIL-CREDA*         0.679 ± 0.000     0.810 ± 0.000     0.586 ± 0.000     0.744 ± 0.000     0.822 ± 0.000     0.728 ± 0.000     0.728
MIL-CREDA          

## 4 · Cada método contra su propio piso

**Qué mide:** el cambio en la razón y en la separabilidad respecto del mismo
método con el término de adaptación apagado. **Por qué:** una distancia en un
embedding no tiene significado absoluto, así que lo que lleva información es
**cuánto se movió**. **Un cambio negativo en la razón es mejor**: los dominios se
juntaron, clase por clase, más de lo que se juntaron las clases entre sí.

In [9]:
compared = latent.against_floor(readings)
print(f"{'método':<14}{'piso':<14}{'transf.':<9}{'sem.':>5}"
      f"{'razón':>9}{'piso':>9}{'cambio':>9}{'separab.':>10}{'cambio':>9}")
for row in compared:
    print(f"{config.NAME_OF[row['arm']]:<14}{config.NAME_OF[row['floor']]:<14}"
          f"{row['transfer']:<9}{row['seed']:>5}"
          f"{row['ratio']:>9.3f}{row['floorRatio']:>9.3f}{row['ratioChange']:>+9.3f}"
          f"{row['separability']:>10.3f}{row['separabilityChange']:>+9.3f}")

método        piso          transf.   sem.    razón     piso   cambio  separab.   cambio
CREDA*        Baseline      M->S         0    1.230    1.205   +0.025     0.954   +0.007
CREDA*        Baseline      M->U         0    0.586    0.598   -0.012     0.921   -0.008
CREDA*        Baseline      S->M         0    1.521    1.474   +0.047     0.992   -0.001
CREDA*        Baseline      S->U         0    1.343    1.320   +0.022     0.974   -0.007
CREDA*        Baseline      U->M         0    0.555    0.530   +0.025     0.876   +0.025
CREDA*        Baseline      U->S         0    1.195    1.205   -0.010     0.887   -0.007
CREDA         Baseline      M->S         0    1.218    1.205   +0.013     0.970   +0.023
CREDA         Baseline      M->U         0    0.691    0.598   +0.093     0.942   +0.012
CREDA         Baseline      S->M         0    1.780    1.474   +0.307     0.994   +0.001
CREDA         Baseline      S->U         0    1.451    1.320   +0.131     0.986   +0.005
CREDA         Baselin

## 5 · ¿Son redundantes los dos pisos?

**Qué mide:** si `Baseline` y `MIL-Baseline`, dibujados los dos a nivel de
instancia, representan el mismo espacio. **Por qué:** de la respuesta depende si
una columna de la grilla sobra. Y no se puede contestar leyendo el código: los dos
entrenan el mismo codificador con objetivos distintos — entropía cruzada por
instancia contra entropía cruzada por bolsa a través del agrupamiento por
atención — así que sus embeddings de instancia no tienen por qué coincidir.

Se compara la **geometría**, no las coordenadas: dos codificadores pueden aprender
el mismo espacio rotado, y comparar coordenadas llamaría distintos a dos espacios
idénticos.

In [10]:
transfers = tables.best_transfers(runs)
pisos = latent.floors_agree(transfers, seed, device)
print(pisos["detail"])
print()
for fila in pisos["byTransfer"]:
    print(f"  {fila['transfer']:<7} razón A={fila['ratio']['A']:.3f} "
          f"B={fila['ratio']['B']:.3f} (dif {fila['ratioGap']:.3f})   "
          f"separab. A={fila['separability']['A']:.3f} "
          f"B={fila['separability']['B']:.3f} (dif {fila['separabilityGap']:.3f})")

paneles = ([a for a in config.LATENT_PANELS if a != "B"] if pisos["agree"]
           else config.LATENT_PANELS)

Baseline y MIL-Baseline NO representan lo mismo a nivel de instancia: la razón entre distancias difiere hasta 0.381 y la separabilidad hasta 0.070, contra una tolerancia de 0.05. Sacar una de las dos columnas pierde el piso de esa familia.

  U->M    razón A=0.530 B=0.673 (dif 0.143)   separab. A=0.851 B=0.921 (dif 0.070)
  M->U    razón A=0.598 B=0.979 (dif 0.381)   separab. A=0.930 B=0.989 (dif 0.059)
  S->M    razón A=1.474 B=1.518 (dif 0.044)   separab. A=0.994 B=0.997 (dif 0.004)


## 6 · La grilla

**Qué se mira:** el espacio de representación de cada método, en tres
transferencias. Filas: transferencias. Columnas: el espacio original compartido
del par de dominios, y después un método cada una. El color es la clase; el
marcador es el dominio — círculos fuente, triángulos destino.

**Qué se busca:** que un color quede junto a sí mismo cruzando marcadores —
círculos y triángulos del mismo color mezclados — **sin** que todos los colores se
junten entre sí. Lo primero solo es alineación condicional si lo segundo no pasa;
si pasan las dos cosas, el espacio colapsó.

La primera columna es el espacio original compartido: las imágenes mismas, antes
de cualquier modelo. Es compartido porque el preprocesamiento ya lleva los dos
dominios a la misma forma, así que los dos conjuntos de píxeles viven en un mismo
espacio vectorial. Es representativo y no completo: la partición de evaluación
tiene muchas más instancias de las que entran en un panel, así que la muestra se
estratifica por clase.

Los pisos son la referencia: *alineado* no se puede ver sin un *no alineado* al
lado. Cuáles pisos quedan lo decide la medición de arriba, no una suposición.

**Todos los paneles se dibujan a nivel de instancia**, incluidos los métodos que
predicen por bolsa. Todos codifican instancias — ahí aplica la Ec. (13) — así que
es un espacio que todos tienen y la única forma de que todos los paneles lleven la
misma cantidad de puntos. La vista por bolsa no se pierde: las tablas de arriba
miden a cada método en su propia unidad.

Las tres transferencias son las de **mayor exactitud media en destino**, calculadas
de la campaña. Es una elección hecha por el resultado y por eso va declarada al
pie: el espacio latente de una transferencia donde todos quedan cerca del azar es
la foto de un modelo que no aprendió, y de ahí no se lee nada sobre alineación. Lo
que esa elección nunca toca es **qué sorteo** se muestra: eso sigue siendo la
semilla de exhibición.

In [11]:
sello = ("" if len(config.SEEDS) >= len(config.FULL_SEEDS)
         else " · PILOTO, no es un resultado")
grid = latent.latent_grid(
    config.RESULTS / "latent" / "grid.png", paneles, transfers, seed, device,
    caption=(f"semilla {seed} de {len(config.SEEDS)} · {config.EPOCHS} épocas · "
             f"{config.LATENT_POINTS} instancias por dominio y panel, estratificadas "
             f"por clase · transferencias elegidas por "
             f"{config.FIGURE_TRANSFER_RULE} · {config.REVISION}{sello}"))
print("escrita:", grid.relative_to(REPOSITORY))
print()
print("Conclusión:", tables.conclusion_geometry(readings))

/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


...

/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


escrita: MIL-CREDA/Results/Benchmark/latent/grid.png

Conclusión: Cada método contra su propio piso, transferencia por transferencia. «Alinea» = la razón bajó; «colapsa» = la razón no bajó y la distancia entre clases cayó más del 10%.

Método        piso            alinea  plano  empeora  colapsa
CREDA         Baseline             0      1        5        3
MIL-CREDA     MIL-Baseline         2      1        3        4
MIL-CREDA*    MIL-Baseline         4      1        1        2
MIL-CREDA**   MIL-Baseline         3      2        1        2
MIL-CREDA-U   MIL-Baseline         0      0        6        4
MIL-CREDA-A   MIL-Baseline         1      0        5        5
MIL-CREDA-K   MIL-Baseline         1      1        4        5
CREDA*        Baseline             0      2        4        1

Lo que carga peso no es el promedio sino que las transferencias coincidan: un método que alinea en una y empeora en otra no está diciendo nada todavía.
Piloto: son estimaciones puntuales, no resultados.


## 7 · La correspondencia, sujeto por sujeto

**Qué se mira:** para una bolsa de destino de cada clase, cuál es la bolsa de
fuente más cercana según el kernel de bolsas. Filas: las mismas tres
transferencias. Columnas: `MIL-Baseline`, `MIL-CREDA*` y `MIL-CREDA`.

**Qué se busca:** que la línea sea llena, es decir que la vecina de fuente sea de
la clase correcta. El azar es 1/10. Y sobre todo, que la columna de la derecha
separe de la del medio: esas dos se diferencian **solo** en el término local, así
que esa distancia es lo único que lo aísla.

**Por qué esas tres y no las mejores del ranking:** esta figura es sobre el
término local, y ese es el peldaño donde vive — el piso, el mismo método sin el
término, y el completo. Así la figura **puede salir mal**: si la columna del medio
empareja igual de bien que la de la derecha, el término local no está haciendo
nada visible, y eso es un hallazgo que la figura tiene permitido alcanzar.

Se destaca **la misma bolsa de cada clase en todos los paneles de una fila**, y es
la **mediana** de su clase por masa de correspondencia, nunca la mejor: la mejor
bolsa de una clase empareja limpio bajo cualquier método, incluido el piso que no
aprendió ninguna correspondencia, y una figura que no puede salir mal no está
midiendo nada.

La vecina se calcula con el kernel de bolsas **en el espacio de representación** —
nunca euclídea, que el método no usa, y nunca en la proyección, que ilustraría a
UMAP. El par se une con una línea para que la correspondencia sobreviva a la
distorsión de la proyección.

In [12]:
correspondencia = latent.correspondence_grid(
    config.RESULTS / "latent" / "correspondence.png", config.BAG_PANELS, transfers,
    seed, device,
    caption=(f"semilla {seed} · las bolsas de fuente son sujetos de entrenamiento y "
             f"las de destino sujetos de evaluación · transferencias elegidas por "
             f"{config.FIGURE_TRANSFER_RULE} · {config.REVISION}{sello}"))
print("escrita:", correspondencia["path"].relative_to(REPOSITORY))
print()
print(f"{'método':<14}{'transf.':<9}{'aciertos':>10}{'masa':>9}")
for fila in correspondencia["scored"]:
    print(f"{config.NAME_OF[fila['arm']]:<14}{fila['transfer']:<9}"
          f"{fila['hits']:>4}/{fila['classes']:<5}{fila['mass']:>9.3f}")
print()
print("Conclusión:", tables.conclusion_correspondence(correspondencia["scored"]))

/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


escrita: MIL-CREDA/Results/Benchmark/latent/correspondence.png

método        transf.    aciertos     masa
MIL-Baseline  U->M        7/10       0.766
MIL-CREDA*    U->M        9/10       0.792
MIL-CREDA     U->M       10/10       0.851
MIL-Baseline  M->U        2/10       0.434
MIL-CREDA*    M->U        3/10       0.501
MIL-CREDA     M->U        2/10       0.423
MIL-Baseline  S->M        2/10       0.300
MIL-CREDA*    S->M        2/10       0.265
MIL-CREDA     S->M        2/10       0.268

Conclusión: Aciertos por transferencia (azar 0.100): MIL-Baseline U->M 7/10, M->U 2/10, S->M 2/10 · MIL-CREDA* U->M 9/10, M->U 3/10, S->M 2/10 · MIL-CREDA U->M 10/10, M->U 2/10, S->M 2/10. El término local, aislado como MIL-CREDA contra MIL-CREDA*: las transferencias no coinciden — 1 a favor, 1 en contra, 1 planas (M->U -0.078, S->M +0.003, U->M +0.060). Un promedio acá diría 'no hace nada' y estaría tapando que una transferencia sí se movió. Piloto: son estimaciones puntuales, no resultados.


## 8 · El registro

Generado junto con los resultados y nunca escrito a mano, por la misma razón que
en la fase uno: una conclusión escrita debajo de una figura se fija, la figura se
vuelve a generar con otros datos y la frase se queda.

In [13]:
output = config.RESULTS / "latent.json"
output.write_text(json.dumps({
    "revision": config.REVISION,
    "displaySeed": seed,
    "figureTransfers": transfers,
    "figureTransferRule": config.FIGURE_TRANSFER_RULE,
    "gridPanels": paneles,
    "gridUnit": config.LATENT_UNIT,
    "bagPanels": config.BAG_PANELS,
    "floorsAgree": {k: v for k, v in pisos.items() if k != "byTransfer"},
    "readings": readings,
    "againstFloor": compared,
    "correspondence": correspondencia["scored"],
    "conclusions": {
        "geometry": tables.conclusion_geometry(readings),
        "correspondence": tables.conclusion_correspondence(correspondencia["scored"]),
    },
    "figures": [str(grid.relative_to(REPOSITORY)),
                str(correspondencia["path"].relative_to(REPOSITORY))],
}, indent=2), encoding="utf-8")

(config.RESULTS / "latent.md").write_text("\n\n".join([
    f"# Fase dos — {config.REVISION}",
    "## 1 · Razón entre distancias (más bajo es mejor, si la de entre clases no cayó)",
    tables.render_readings(readings, "geometry.ratio",
                           "razón: misma clase entre dominios / entre clases",
                           markdown=True),
    tables.conclusion_geometry(readings),
    "## 2 · Separabilidad de dominio (más cerca de 0,500 es mejor)",
    tables.render_readings(readings, "domainSeparability",
                           "exactitud de un clasificador de dominio", markdown=True),
    "## 3 · Masa en la clase verdadera (más alto es mejor, azar 0,100)",
    tables.render_readings(readings, "correspondence.massOnTrueClass",
                           "masa en la clase verdadera", markdown=True),
    tables.conclusion_correspondence(correspondencia["scored"]),
]), encoding="utf-8")
print("escrito", output.relative_to(REPOSITORY))

escrito MIL-CREDA/Results/Benchmark/latent.json


In [14]:
# El sello: contra qué código corrió este informe. Sin él, un informe viejo y uno
# recién generado se ven idénticos y el viejo se sigue creyendo.
from MIL_CREDA_Benchmark import report_digest

print(report_digest.stamp())

SOURCES-SHA256 0cf518226878861386cf8b7edb096149e50a100ef0391d65b1599463edcae5bb
